# Adaptive Ensure/lick convolutional GLM

This notebook uses the **complete duration of every Doric recording**. Runs without Ensure deliveries help identify the lick-related lifetime component; reward runs identify the Ensure response and how its waveform changes across deliveries.

The first processing cell is slow once. It creates a reusable cache. All model settings and plots after that operate on the cache.

In [ ]:
from datetime import date, datetime
from pathlib import Path
from time import perf_counter
import csv
import matplotlib.pyplot as plt
import numpy as np
from openpyxl import load_workbook
from fluopulse_analysis import find_doric_files, read_doric

## Configuration and full-recording cache

Set `REBUILD_CACHE = True` the first time, or whenever source data change. The cache stores fitted lifetime plus lick and Ensure events; it does not load raw fluorescence waveforms.

In [ ]:
SESSION_TABLE = Path.home() / "Documents" / "SC_FLiPmice_2025.xlsx"
DORIC_ROOT = Path(r"Z:\FLIM FLIP")
OUTPUT_FOLDER = Path.cwd() / "analysis_output_adaptive_glm"
CACHE_FILE = OUTPUT_FOLDER / "full_recording_glm_cache.npz"
REBUILD_CACHE = False
SAMPLE_INTERVAL_S = 0.1
BASELINE_REFERENCE_S = 30.0
print(f"Doric root available: {DORIC_ROOT.is_dir()} ({DORIC_ROOT})")

In [ ]:
def normalize_date(value):
    if isinstance(value, (date, datetime)):
        return value.strftime("%y%m%d")
    return str(int(value)).zfill(6)

def recording_label(path, mouse, session_date):
    stem = path.stem
    for prefix in (f"{mouse}_{session_date}_", f"{session_date}_{mouse}_"):
        if stem.casefold().startswith(prefix.casefold()):
            return stem[len(prefix):]
    return stem

workbook = load_workbook(SESSION_TABLE, data_only=True, read_only=True)
worksheet = workbook.active
rows = list(worksheet.iter_rows(values_only=True))
workbook.close()
headers = [str(value).strip() if value is not None else "" for value in rows[0]]
session_records = []
for row in rows[1:]:
    if not row or row[0] is None:
        continue
    mouse = str(row[0]).strip()
    for column, value in zip(headers[1:], row[1:], strict=False):
        if value is not None:
            session_records.append({"mouse": mouse, "date": normalize_date(value), "source_column": column})
print(f"Loaded {len(session_records)} mouse-date sessions")

In [ ]:
if REBUILD_CACHE or not CACHE_FILE.is_file():
    cached_recordings = []
    qc_rows = []
    started = perf_counter()
    print(f"Reading {len(session_records)} mouse-date sessions", flush=True)
    for session_index, record in enumerate(session_records, start=1):
        mouse, session_date = record["mouse"], record["date"]
        print(f"[{session_index}/{len(session_records)}] {mouse} {session_date}", flush=True)
        doric_files = find_doric_files(mouse, session_date, doric_root=DORIC_ROOT)
        if not doric_files:
            qc_rows.append({"mouse": mouse, "date": session_date, "recording": "", "status": "missing .doric"})
            continue
        for file_index, doric_file in enumerate(doric_files, start=1):
            label = recording_label(doric_file, mouse, session_date)
            print(f"    ({file_index}/{len(doric_files)}) {label}", flush=True)
            try:
                recording = read_doric(doric_file)
                finite = np.isfinite(recording.time) & np.isfinite(recording.tau_ns)
                if np.sum(finite) < 10:
                    raise ValueError("fewer than 10 finite lifetime samples")
                source_time = recording.time[finite] - recording.time[finite][0]
                source_tau = recording.tau_ns[finite]
                time = np.arange(0.0, source_time[-1] + SAMPLE_INTERVAL_S / 2, SAMPLE_INTERVAL_S)
                tau = np.interp(time, source_time, source_tau, left=np.nan, right=np.nan)
                baseline = time < min(BASELINE_REFERENCE_S, max(SAMPLE_INTERVAL_S, time[-1] / 4))
                reference_tau = float(np.nanmedian(tau[baseline]))
                lick_pulses = recording.digital_pulses.get("licking")
                ensure_pulses = recording.digital_pulses.get("ensure")
                lick_times = np.array([]) if lick_pulses is None else lick_pulses.onset_times - recording.time[finite][0]
                ensure_times = np.array([]) if ensure_pulses is None else ensure_pulses.onset_times - recording.time[finite][0]
                edges = np.concatenate([time - SAMPLE_INTERVAL_S / 2, [time[-1] + SAMPLE_INTERVAL_S / 2]])
                cached_recordings.append({
                    "mouse": mouse, "date": session_date, "recording": label, "source_file": str(doric_file),
                    "time": time, "delta_tau": tau - reference_tau,
                    "lick_counts": np.histogram(lick_times, bins=edges)[0].astype(float),
                    "ensure_counts": np.histogram(ensure_times, bins=edges)[0].astype(float),
                    "reference_tau": reference_tau, "duration_s": float(time[-1]),
                })
                qc_rows.append({"mouse": mouse, "date": session_date, "recording": label, "status": "ok", "duration_s": time[-1], "lick_count": int(lick_times.size), "ensure_count": int(ensure_times.size)})
            except Exception as error:
                qc_rows.append({"mouse": mouse, "date": session_date, "recording": label, "status": f"error: {error}"})
                print(f"        ERROR: {error}", flush=True)
    if not cached_recordings:
        raise ValueError("No usable Doric recordings were found")
    maximum_length = max(item["time"].size for item in cached_recordings)
    def padded(field):
        result = np.full((len(cached_recordings), maximum_length), np.nan)
        for index, item in enumerate(cached_recordings):
            result[index, :item[field].size] = item[field]
        return result
    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(CACHE_FILE, time_s=padded("time"), delta_tau_ns=padded("delta_tau"), lick_counts_per_bin=padded("lick_counts"), ensure_counts_per_bin=padded("ensure_counts"), bin_width_s=np.asarray(SAMPLE_INTERVAL_S), mouse=np.asarray([item["mouse"] for item in cached_recordings]), session_date=np.asarray([item["date"] for item in cached_recordings]), recording=np.asarray([item["recording"] for item in cached_recordings]), source_file=np.asarray([item["source_file"] for item in cached_recordings]), reference_tau_ns=np.asarray([item["reference_tau"] for item in cached_recordings]), duration_s=np.asarray([item["duration_s"] for item in cached_recordings]))
    fieldnames = list(dict.fromkeys(key for row in qc_rows for key in row))
    with (OUTPUT_FOLDER / "full_recording_cache_qc.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames); writer.writeheader(); writer.writerows(qc_rows)
    print(f"Cached {len(cached_recordings)} complete recordings in {perf_counter() - started:.1f} s")
else:
    print(f"Using existing cache: {CACHE_FILE.resolve()}")

## Fast model configuration

The Gaussian convolutional GLM includes a lick kernel, a common Ensure kernel, a first-delivery-specific kernel, and linear plus quadratic interactions across subsequent deliveries. This separates the large initial state transition from early, middle, and late responses after the first delivery. Cross-validation holds out complete recordings.

In [ ]:
MOUSE_FILTER = []
DATE_FILTER = []
RECORDING_TEXT = ""
LICK_KERNEL_S = 10.0
ENSURE_KERNEL_S = 45.0
LICK_BASIS_COUNT = 8
ENSURE_BASIS_COUNT = 14
RIDGE_ALPHAS = np.logspace(-3, 4, 12)
CV_GROUPING = "mouse"  # Falls back to recording when only one mouse is selected

with np.load(CACHE_FILE, allow_pickle=False) as cache:
    time_matrix = cache["time_s"]
    delta_tau_matrix = cache["delta_tau_ns"]
    lick_matrix = cache["lick_counts_per_bin"]
    ensure_matrix = cache["ensure_counts_per_bin"]
    bin_width_s = float(cache["bin_width_s"])
    mice = cache["mouse"].astype(str)
    dates = cache["session_date"].astype(str)
    recording_names = cache["recording"].astype(str)
selected = np.ones(mice.size, dtype=bool)
if MOUSE_FILTER: selected &= np.isin(mice, np.asarray(MOUSE_FILTER, dtype=str))
if DATE_FILTER: selected &= np.isin(dates, np.asarray(DATE_FILTER, dtype=str))
if RECORDING_TEXT: selected &= np.char.find(np.char.lower(recording_names), RECORDING_TEXT.casefold()) >= 0
time_matrix, delta_tau_matrix = time_matrix[selected], delta_tau_matrix[selected]
lick_matrix, ensure_matrix = lick_matrix[selected], ensure_matrix[selected]
mice, dates, recording_names = mice[selected], dates[selected], recording_names[selected]
print(f"Selected {mice.size} full recordings: {np.sum(np.nansum(ensure_matrix, axis=1) == 0)} without Ensure and {np.sum(np.nansum(ensure_matrix, axis=1) > 0)} with Ensure")

In [ ]:
def raised_cosine_basis(duration_s, count, dt):
    lag = np.arange(0.0, duration_s + dt / 2, dt)
    centers = np.linspace(0.0, duration_s, count)
    width = centers[1] - centers[0] if count > 1 else duration_s
    basis = np.zeros((lag.size, count))
    for column, center in enumerate(centers):
        distance = (lag - center) / max(width, dt)
        inside = np.abs(distance) <= 1
        basis[inside, column] = 0.5 * (1 + np.cos(np.pi * distance[inside]))
    return lag, basis

def convolve_basis(events, basis):
    return np.column_stack([np.convolve(events, basis[:, column], mode="full")[:events.size] for column in range(basis.shape[1])])

lick_lag, lick_basis = raised_cosine_basis(LICK_KERNEL_S, LICK_BASIS_COUNT, bin_width_s)
ensure_lag, ensure_basis = raised_cosine_basis(ENSURE_KERNEL_S, ENSURE_BASIS_COUNT, bin_width_s)
designs, outcomes, groups = [], [], []
recording_designs = []
use_mouse_cv = CV_GROUPING.casefold() == "mouse" and np.unique(mice).size > 1
for recording_index in range(mice.size):
    valid = np.isfinite(time_matrix[recording_index]) & np.isfinite(delta_tau_matrix[recording_index])
    time = time_matrix[recording_index, valid]
    outcome = delta_tau_matrix[recording_index, valid]
    licks = np.nan_to_num(lick_matrix[recording_index, valid])
    ensures = np.nan_to_num(ensure_matrix[recording_index, valid])
    event_indices = np.flatnonzero(ensures > 0)
    first_ensure = np.zeros_like(ensures)
    progress_train = np.zeros_like(ensures)
    if event_indices.size:
        first_ensure[event_indices[0]] = ensures[event_indices[0]]
    if event_indices.size > 1:
        progress_train[event_indices[1:]] = np.linspace(0.0, 1.0, event_indices.size - 1)
    lick_design = convolve_basis(licks, lick_basis)
    ensure_design = convolve_basis(ensures, ensure_basis)
    first_ensure_design = convolve_basis(first_ensure, ensure_basis)
    ensure_linear = convolve_basis(ensures * progress_train, ensure_basis)
    ensure_quadratic = convolve_basis(ensures * progress_train ** 2, ensure_basis)
    normalized_time = time / max(time[-1], bin_width_s)
    design = np.column_stack([np.ones(time.size), normalized_time, normalized_time ** 2, lick_design, ensure_design, first_ensure_design, ensure_linear, ensure_quadratic])
    cv_label = mice[recording_index] if use_mouse_cv else str(recording_index)
    designs.append(design); outcomes.append(outcome); groups.append(np.full(time.size, cv_label, dtype=object)); recording_designs.append((time, design, outcome))
X = np.vstack(designs); y = np.concatenate(outcomes); group = np.concatenate(groups)
column_scale = np.sqrt(np.mean(X ** 2, axis=0)); column_scale[column_scale == 0] = 1
Xs = X / column_scale
penalty = np.eye(X.shape[1]); penalty[0, 0] = 0
unique_groups = np.unique(group)
cv_scores = []
for alpha in RIDGE_ALPHAS:
    fold_scores = []
    for held_out in unique_groups:
        train, test = group != held_out, group == held_out
        beta_s = np.linalg.solve(Xs[train].T @ Xs[train] + alpha * penalty, Xs[train].T @ y[train])
        prediction = Xs[test] @ beta_s
        denominator = np.sum((y[test] - np.mean(y[test])) ** 2)
        fold_scores.append(np.nan if denominator == 0 else 1 - np.sum((y[test] - prediction) ** 2) / denominator)
    cv_scores.append(np.nanmean(fold_scores))
best_alpha = float(RIDGE_ALPHAS[int(np.nanargmax(cv_scores))])
beta_scaled = np.linalg.solve(Xs.T @ Xs + best_alpha * penalty, Xs.T @ y)
beta = beta_scaled / column_scale
cv_description = "mouse" if use_mouse_cv else "recording"
best_cv_score = float(np.nanmax(cv_scores))
print(f"Best ridge alpha: {best_alpha:g}; mean held-{cv_description}-out R2: {best_cv_score:.3f}")
if np.sum(np.nansum(ensure_matrix, axis=1) > 0) < 2:
    print("WARNING: Fewer than two Ensure recordings are selected; adaptive Ensure generalization cannot yet be validated.")
if best_cv_score < 0:
    print("WARNING: Cross-validated R2 is below zero. Treat the kernels as exploratory; the model does not yet generalize better than the held-out mean.")

In [ ]:
offset = 3
lick_coefficients = beta[offset:offset + LICK_BASIS_COUNT]
offset += LICK_BASIS_COUNT
ensure_base = beta[offset:offset + ENSURE_BASIS_COUNT]
first_ensure_delta = beta[offset + ENSURE_BASIS_COUNT:offset + 2 * ENSURE_BASIS_COUNT]
ensure_linear = beta[offset + 2 * ENSURE_BASIS_COUNT:offset + 3 * ENSURE_BASIS_COUNT]
ensure_quadratic = beta[offset + 3 * ENSURE_BASIS_COUNT:offset + 4 * ENSURE_BASIS_COUNT]
lick_kernel = lick_basis @ lick_coefficients
progress_levels = [0.0, 0.5, 1.0]
subsequent_group_names = ["Early subsequent", "Middle subsequent", "Late subsequent"]
group_names = ["First", *subsequent_group_names]
group_colors = ["tab:red", "tab:blue", "tab:orange", "tab:green"]
first_ensure_kernel = ensure_basis @ (ensure_base + first_ensure_delta)
subsequent_ensure_kernels = [ensure_basis @ (ensure_base + progress * ensure_linear + progress ** 2 * ensure_quadratic) for progress in progress_levels]
ensure_kernels = [first_ensure_kernel, *subsequent_ensure_kernels]
example_index = int(np.nanargmax(np.nansum(ensure_matrix, axis=1)))
example_time, example_design, example_outcome = recording_designs[example_index]
example_prediction = example_design @ beta
figure, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
axes[0, 0].plot(lick_lag, lick_kernel, color="tab:green")
axes[0, 0].axhline(0, color="black", linewidth=0.8); axes[0, 0].set(title="Estimated lick kernel", xlabel="Lag from lick (s)", ylabel="Delta tau contribution (ns)")
for kernel, label, color in zip(ensure_kernels, group_names, group_colors):
    axes[0, 1].plot(ensure_lag, kernel, label=label, color=color)
axes[0, 1].axhline(0, color="black", linewidth=0.8); axes[0, 1].set(title="Ensure kernel with adaptation", xlabel="Lag from Ensure (s)", ylabel="Delta tau contribution (ns)"); axes[0, 1].legend()
axes[1, 0].semilogx(RIDGE_ALPHAS, cv_scores, marker="o"); axes[1, 0].axvline(best_alpha, color="tab:orange", linestyle="--"); axes[1, 0].set(title=f"Held-{cv_description}-out cross-validation", xlabel="Ridge alpha", ylabel="Mean R2")
axes[1, 1].plot(example_time, example_outcome, color="0.7", linewidth=0.8, label="Observed")
axes[1, 1].plot(example_time, example_prediction, color="tab:blue", linewidth=1.2, label="GLM prediction")
axes[1, 1].set(title=f"Example: {mice[example_index]} {recording_names[example_index]}", xlabel="Recording time (s)", ylabel="Delta tau (ns)"); axes[1, 1].legend()
plt.show()

In [ ]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
figure.savefig(OUTPUT_FOLDER / "adaptive_ensure_glm.png", dpi=200)
np.savez_compressed(OUTPUT_FOLDER / "adaptive_ensure_glm_results.npz", lick_lag_s=lick_lag, lick_kernel_ns=lick_kernel, ensure_lag_s=ensure_lag, ensure_kernel_first_ns=ensure_kernels[0], ensure_kernel_early_subsequent_ns=ensure_kernels[1], ensure_kernel_middle_subsequent_ns=ensure_kernels[2], ensure_kernel_late_subsequent_ns=ensure_kernels[3], ridge_alphas=RIDGE_ALPHAS, cv_scores=np.asarray(cv_scores), best_alpha=np.asarray(best_alpha), coefficients=beta)
with (OUTPUT_FOLDER / "adaptive_ensure_glm_summary.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["best_alpha", "cross_validated_r2", "recording_count", "no_ensure_recordings", "ensure_recordings"]); writer.writeheader(); writer.writerow({"best_alpha": best_alpha, "cross_validated_r2": float(np.nanmax(cv_scores)), "recording_count": mice.size, "no_ensure_recordings": int(np.sum(np.nansum(ensure_matrix, axis=1) == 0)), "ensure_recordings": int(np.sum(np.nansum(ensure_matrix, axis=1) > 0))})
print(f"Saved model outputs to {OUTPUT_FOLDER.resolve()}")

## Interpretation

The first-delivery curve estimates an additional response unique to the initial Ensure transition. Early/middle/late curves describe only subsequent deliveries. A difference between the subsequent curves is evidence of delivery-dependent adaptation within this model. The lick curve is constrained by all recordings, especially the no-Ensure runs.

Because Ensure delivery and licking are behaviorally coupled, coefficient separation is not automatically causal. Regular 15-second delivery intervals also limit identification of response components longer than 15 seconds. Held-recording-out performance and future sessions with variable delivery intervals are important validation.

## Nested-model cross-validation

This comparison separates variance explained by session drift, licking, a common Ensure kernel, the additional first-delivery transition, and adaptation across subsequent deliveries. Each model selects its own ridge penalty using the same held-mouse-out (or recording fallback) folds.

In [ ]:
drift_end = 3
lick_end = drift_end + LICK_BASIS_COUNT
fixed_ensure_end = lick_end + ENSURE_BASIS_COUNT
first_specific_end = fixed_ensure_end + ENSURE_BASIS_COUNT
nested_model_columns = {
    "Drift only": np.arange(drift_end),
    "+ licking": np.arange(lick_end),
    "+ fixed Ensure": np.arange(fixed_ensure_end),
    "+ first-delivery transition": np.arange(first_specific_end),
    "+ subsequent adaptation": np.arange(X.shape[1]),
}

def nested_cv(columns):
    model_X = X[:, columns]
    scale = np.sqrt(np.mean(model_X ** 2, axis=0)); scale[scale == 0] = 1
    model_Xs = model_X / scale
    model_penalty = np.eye(columns.size)
    if columns[0] == 0: model_penalty[0, 0] = 0
    alpha_scores = []
    for alpha in RIDGE_ALPHAS:
        fold_scores = []
        for held_out in unique_groups:
            train, test = group != held_out, group == held_out
            if not np.any(train) or not np.any(test):
                continue
            coefficients = np.linalg.solve(model_Xs[train].T @ model_Xs[train] + alpha * model_penalty, model_Xs[train].T @ y[train])
            prediction = model_Xs[test] @ coefficients
            denominator = np.sum((y[test] - np.mean(y[test])) ** 2)
            if denominator > 0: fold_scores.append(1 - np.sum((y[test] - prediction) ** 2) / denominator)
        alpha_scores.append(np.nan if not fold_scores else np.mean(fold_scores))
    best_index = int(np.nanargmax(alpha_scores))
    return float(RIDGE_ALPHAS[best_index]), float(alpha_scores[best_index]), np.asarray(alpha_scores)

nested_results = []
for model_name, columns in nested_model_columns.items():
    model_alpha, model_score, model_curve = nested_cv(columns)
    nested_results.append({"model": model_name, "best_alpha": model_alpha, "cross_validated_r2": model_score})
    print(nested_results[-1])
nested_scores = np.asarray([row["cross_validated_r2"] for row in nested_results])
incremental_scores = np.diff(np.concatenate([[0.0], nested_scores]))
nested_figure, nested_axis = plt.subplots(figsize=(9, 5), constrained_layout=True)
bars = nested_axis.bar(np.arange(len(nested_results)), nested_scores, color=["0.6", "tab:green", "tab:orange", "tab:red", "tab:blue"])
nested_axis.axhline(0, color="black", linewidth=0.8)
nested_axis.set_xticks(np.arange(len(nested_results)), [row["model"] for row in nested_results], rotation=15, ha="right")
nested_axis.set_ylabel(f"Held-{cv_description}-out R2")
nested_axis.set_title("Incremental predictive value of GLM components")
for bar, score, increment in zip(bars, nested_scores, incremental_scores):
    nested_axis.text(bar.get_x() + bar.get_width() / 2, score, f"{score:.3f}\nDelta {increment:+.3f}", ha="center", va="bottom" if score >= 0 else "top")
plt.show()

## Empirical first and subsequent responses before the next delivery

The first Ensure delivery is retained as its own group. Remaining deliveries are divided into early, middle, and late thirds within each recording. Each trace is locally baseline-corrected using −5 to −0.5 seconds, restricted to 0–15 seconds, averaged within recording, then within mouse. This preserves the initial transition while preventing mice with more subsequent deliveries from dominating the mean.

In [ ]:
EMPIRICAL_WINDOW_S = (0.0, 15.0)
EMPIRICAL_BASELINE_S = (-5.0, -0.5)
def delivery_group_indices(event_count):
    groups = np.zeros(event_count, dtype=int)
    subsequent_count = max(0, event_count - 1)
    if subsequent_count:
        groups[1:] = 1 + np.minimum(2, (3 * np.arange(subsequent_count) // subsequent_count)).astype(int)
    return groups
post_samples = int(round((EMPIRICAL_WINDOW_S[1] - EMPIRICAL_WINDOW_S[0]) / bin_width_s))
baseline_start = int(round(abs(EMPIRICAL_BASELINE_S[0]) / bin_width_s))
baseline_end = int(round(abs(EMPIRICAL_BASELINE_S[1]) / bin_width_s))
empirical_time = np.arange(post_samples) * bin_width_s
recording_traces = {name: [] for name in group_names}
recording_trace_mice = {name: [] for name in group_names}
for recording_index in range(mice.size):
    event_indices = np.flatnonzero(np.nan_to_num(ensure_matrix[recording_index]) > 0)
    if event_indices.size == 0:
        continue
    event_groups = delivery_group_indices(event_indices.size)
    per_group = {name: [] for name in group_names}
    for event_index, group_index in zip(event_indices, event_groups):
        if event_index - baseline_start < 0 or event_index + post_samples > delta_tau_matrix.shape[1]:
            continue
        baseline_values = delta_tau_matrix[recording_index, event_index - baseline_start:event_index - baseline_end]
        response = delta_tau_matrix[recording_index, event_index:event_index + post_samples]
        if np.mean(np.isfinite(response)) < 0.8 or not np.any(np.isfinite(baseline_values)):
            continue
        per_group[group_names[group_index]].append(response - np.nanmean(baseline_values))
    for name in group_names:
        if per_group[name]:
            recording_traces[name].append(np.nanmean(np.vstack(per_group[name]), axis=0))
            recording_trace_mice[name].append(mice[recording_index])

def mouse_balanced_summary(traces, trace_mice):
    traces = np.vstack(traces); trace_mice = np.asarray(trace_mice)
    mouse_means = np.vstack([np.nanmean(traces[trace_mice == mouse], axis=0) for mouse in np.unique(trace_mice)])
    mean = np.nanmean(mouse_means, axis=0)
    sem = np.zeros_like(mean) if mouse_means.shape[0] == 1 else np.nanstd(mouse_means, axis=0, ddof=1) / np.sqrt(mouse_means.shape[0])
    return mouse_means, mean, sem

empirical_summaries = {}
empirical_figure, empirical_axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True, constrained_layout=True)
for name, color, glm_kernel in zip(group_names, group_colors, ensure_kernels):
    if not recording_traces[name]:
        continue
    mouse_means, mean, sem = mouse_balanced_summary(recording_traces[name], recording_trace_mice[name])
    empirical_summaries[name] = {"mouse_means": mouse_means, "mean": mean, "sem": sem}
    empirical_axes[0].plot(empirical_time, mean, color=color, linewidth=2, label=f"{name} (mice={mouse_means.shape[0]})")
    empirical_axes[0].fill_between(empirical_time, mean - sem, mean + sem, color=color, alpha=0.2)
    empirical_axes[1].plot(empirical_time, mean, color=color, linewidth=2, label=f"{name} empirical")
    comparable_glm = np.interp(empirical_time, ensure_lag, glm_kernel)
    empirical_axes[1].plot(empirical_time, comparable_glm - comparable_glm[0], color=color, linestyle="--", linewidth=1.5, label=f"{name} GLM")
for axis in empirical_axes:
    axis.axhline(0, color="black", linewidth=0.8); axis.axvline(0, color="tab:orange", linestyle="--")
    axis.set_xlabel("Time from Ensure delivery (s)"); axis.set_ylabel("Locally baseline-corrected delta tau (ns)"); axis.legend()
empirical_axes[0].set_title("Empirical response before the next delivery")
empirical_axes[1].set_title("Empirical response versus fitted GLM kernel")
plt.show()

In [ ]:
nested_figure.savefig(OUTPUT_FOLDER / "adaptive_glm_nested_cross_validation.png", dpi=200)
empirical_figure.savefig(OUTPUT_FOLDER / "empirical_ensure_first_15s.png", dpi=200)
with (OUTPUT_FOLDER / "adaptive_glm_nested_cross_validation.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(nested_results[0])); writer.writeheader(); writer.writerows(nested_results)
empirical_output = {"relative_time_s": empirical_time}
for name, summary in empirical_summaries.items():
    key = name.casefold().replace(" ", "_"); empirical_output[f"{key}_mouse_means"] = summary["mouse_means"]; empirical_output[f"{key}_mean"] = summary["mean"]; empirical_output[f"{key}_sem"] = summary["sem"]
np.savez_compressed(OUTPUT_FOLDER / "empirical_ensure_first_15s.npz", **empirical_output)
print(f"Saved nested-model and empirical-response outputs to {OUTPUT_FOLDER.resolve()}")

## Two-stage constrained lick/consumption model

The joint GLM can divide correlated variance arbitrarily between licking and Ensure TTLs. This version estimates a general lick kernel using only no-Ensure recordings and freezes it. It then adds a first-Ensure state-transition kernel plus reward-associated licking/consumption kernels. The first transition is separate from adaptation across subsequent deliveries.

In [ ]:
sample_recording = np.concatenate([np.full(outcome.size, index, dtype=int) for index, outcome in enumerate(outcomes)])
ensure_recording = np.nansum(ensure_matrix, axis=1) > 0
no_ensure_samples = ~ensure_recording[sample_recording]
lick_columns = np.arange(3, 3 + LICK_BASIS_COUNT)
dry_model_columns = np.arange(3 + LICK_BASIS_COUNT)
def fit_ridge_raw(design, outcome, columns, alpha, sample_mask):
    model_design = design[sample_mask][:, columns]; model_outcome = outcome[sample_mask]
    scale = np.sqrt(np.mean(model_design ** 2, axis=0)); scale[scale == 0] = 1
    scaled = model_design / scale; ridge_penalty = np.eye(columns.size)
    if columns[0] == 0: ridge_penalty[0, 0] = 0
    return np.linalg.solve(scaled.T @ scaled + alpha * ridge_penalty, scaled.T @ model_outcome) / scale
dry_alpha_scores = []
dry_groups = np.unique(group[no_ensure_samples])
for alpha in RIDGE_ALPHAS:
    scores = []
    for held_out in dry_groups:
        train = no_ensure_samples & (group != held_out); test = no_ensure_samples & (group == held_out)
        if not np.any(train) or not np.any(test): continue
        coefficients = fit_ridge_raw(X, y, dry_model_columns, alpha, train)
        prediction = X[test][:, dry_model_columns] @ coefficients
        denominator = np.sum((y[test] - np.mean(y[test])) ** 2)
        if denominator > 0: scores.append(1 - np.sum((y[test] - prediction) ** 2) / denominator)
    dry_alpha_scores.append(np.nan if not scores else np.mean(scores))
best_dry_alpha = float(RIDGE_ALPHAS[int(np.nanargmax(dry_alpha_scores))])
dry_coefficients = fit_ridge_raw(X, y, dry_model_columns, best_dry_alpha, no_ensure_samples)
frozen_lick_coefficients = dry_coefficients[3:]
frozen_lick_prediction = X[:, lick_columns] @ frozen_lick_coefficients
frozen_lick_kernel = lick_basis @ frozen_lick_coefficients
print(f"No-Ensure lick model alpha: {best_dry_alpha:g}; held-{cv_description}-out R2: {np.nanmax(dry_alpha_scores):.3f}")
consumption_designs = []; row_start = 0
for recording_index, outcome in enumerate(outcomes):
    row_end = row_start + outcome.size
    valid = np.isfinite(time_matrix[recording_index]) & np.isfinite(delta_tau_matrix[recording_index])
    licks = np.nan_to_num(lick_matrix[recording_index, valid]); ensures = np.nan_to_num(ensure_matrix[recording_index, valid])
    event_indices = np.flatnonzero(ensures > 0); first_ensure = np.zeros_like(ensures)
    if event_indices.size: first_ensure[event_indices[0]] = ensures[event_indices[0]]
    delivery_number = np.cumsum(ensures > 0); delivery_total = int(np.max(delivery_number)) if delivery_number.size else 0
    reward_available = delivery_number > 0; delivery_progress = np.zeros_like(licks, dtype=float)
    if delivery_total > 1: delivery_progress[reward_available] = (delivery_number[reward_available] - 1) / (delivery_total - 1)
    reward_licks = licks * reward_available
    consumption_designs.append(np.column_stack([X[row_start:row_end, :3], convolve_basis(first_ensure, ensure_basis), convolve_basis(reward_licks, ensure_basis), convolve_basis(reward_licks * delivery_progress, ensure_basis), convolve_basis(reward_licks * delivery_progress ** 2, ensure_basis)]))
    row_start = row_end
constrained_X = np.vstack(consumption_designs)
constrained_specs = {"Drift + frozen lick": np.arange(3), "+ first Ensure transition": np.arange(3 + ENSURE_BASIS_COUNT), "+ fixed consumption": np.arange(3 + 2 * ENSURE_BASIS_COUNT), "+ consumption adaptation": np.arange(constrained_X.shape[1])}


In [ ]:
def constrained_cv(columns, alpha):
    fold_scores = []
    for held_out in unique_groups:
        train, test = group != held_out, group == held_out; dry_train = train & no_ensure_samples
        if not np.any(dry_train) or not np.any(test): continue
        fold_dry = fit_ridge_raw(X, y, dry_model_columns, best_dry_alpha, dry_train)
        fold_lick_prediction = X[:, lick_columns] @ fold_dry[3:]
        stage_coefficients = fit_ridge_raw(constrained_X, y - fold_lick_prediction, columns, alpha, train)
        prediction = fold_lick_prediction[test] + constrained_X[test][:, columns] @ stage_coefficients
        denominator = np.sum((y[test] - np.mean(y[test])) ** 2)
        if denominator > 0: fold_scores.append(1 - np.sum((y[test] - prediction) ** 2) / denominator)
    return np.nan if not fold_scores else float(np.mean(fold_scores))
constrained_results = []
for model_name, columns in constrained_specs.items():
    scores = np.asarray([constrained_cv(columns, alpha) for alpha in RIDGE_ALPHAS]); best_index = int(np.nanargmax(scores))
    constrained_results.append({"model": model_name, "best_alpha": float(RIDGE_ALPHAS[best_index]), "cross_validated_r2": float(scores[best_index])}); print(constrained_results[-1])
final_columns = constrained_specs["+ consumption adaptation"]; final_alpha = constrained_results[-1]["best_alpha"]
first_transition_columns = np.arange(3, 3 + ENSURE_BASIS_COUNT)
full_without_first_columns = np.setdiff1d(final_columns, first_transition_columns)
without_first_scores = np.asarray([constrained_cv(full_without_first_columns, alpha) for alpha in RIDGE_ALPHAS])
without_first_best_index = int(np.nanargmax(without_first_scores))
first_transition_ablation_results = [
    {"model": "Full model without first transition", "best_alpha": float(RIDGE_ALPHAS[without_first_best_index]), "cross_validated_r2": float(without_first_scores[without_first_best_index])},
    {"model": "Full model with first transition", "best_alpha": final_alpha, "cross_validated_r2": constrained_results[-1]["cross_validated_r2"]},
]
first_transition_delta_r2 = first_transition_ablation_results[1]["cross_validated_r2"] - first_transition_ablation_results[0]["cross_validated_r2"]
print(f"Unique held-{cv_description}-out value of first-transition term: delta R2 = {first_transition_delta_r2:+.4f}")
constrained_coefficients = fit_ridge_raw(constrained_X, y - frozen_lick_prediction, final_columns, final_alpha, np.ones(y.size, dtype=bool))
constrained_prediction = frozen_lick_prediction + constrained_X @ constrained_coefficients
first_transition_coefficients = constrained_coefficients[3:3 + ENSURE_BASIS_COUNT]
base = constrained_coefficients[3 + ENSURE_BASIS_COUNT:3 + 2 * ENSURE_BASIS_COUNT]; linear = constrained_coefficients[3 + 2 * ENSURE_BASIS_COUNT:3 + 3 * ENSURE_BASIS_COUNT]; quadratic = constrained_coefficients[3 + 3 * ENSURE_BASIS_COUNT:3 + 4 * ENSURE_BASIS_COUNT]
first_transition_kernel = ensure_basis @ first_transition_coefficients
consumption_progress_levels = [0.0, 1 / 3, 2 / 3, 1.0]
consumption_kernels = [ensure_basis @ (base + progress * linear + progress ** 2 * quadratic) for progress in consumption_progress_levels]
ordinary_lick_on_consumption_lag = np.interp(ensure_lag, lick_lag, frozen_lick_kernel, left=0.0, right=0.0)
total_reward_lick_kernels = [ordinary_lick_on_consumption_lag + kernel for kernel in consumption_kernels]
constrained_scores = np.asarray([row["cross_validated_r2"] for row in constrained_results])
constrained_figure, constrained_axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
constrained_axes[0].plot(lick_lag, frozen_lick_kernel, color="tab:green"); constrained_axes[0].axhline(0, color="black", linewidth=0.8); constrained_axes[0].set(title="Frozen no-Ensure lick kernel", xlabel="Lag from lick (s)", ylabel="Delta tau contribution (ns)")
for kernel, name, color in zip(consumption_kernels, group_names, group_colors): constrained_axes[1].plot(ensure_lag, kernel, color=color, label=f"{name} consumption")
constrained_axes[1].plot(ensure_lag, first_transition_kernel, color="tab:red", linestyle="--", linewidth=2, label="First Ensure TTL transition")
constrained_axes[1].axhline(0, color="black", linewidth=0.8); constrained_axes[1].set(title="First transition and reward-associated lick kernels", xlabel="Lag (s)", ylabel="Delta tau contribution (ns)"); constrained_axes[1].legend()
constrained_axes[2].bar(np.arange(len(constrained_results)), constrained_scores, color=["0.6", "tab:red", "tab:orange", "tab:blue"]); constrained_axes[2].axhline(0, color="black", linewidth=0.8); constrained_axes[2].axhline(best_cv_score, color="tab:purple", linestyle="--", label="Original joint GLM"); constrained_axes[2].set_xticks(np.arange(len(constrained_results)), [row["model"] for row in constrained_results], rotation=15, ha="right"); constrained_axes[2].set_ylabel(f"Held-{cv_description}-out R2"); constrained_axes[2].set_title("Constrained model validation"); constrained_axes[2].legend()
plt.show()
total_lick_figure, total_lick_axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True, constrained_layout=True)
total_lick_axes[0].plot(ensure_lag, ordinary_lick_on_consumption_lag, color="tab:green", linestyle="--", linewidth=2, label="Ordinary lick component")
total_lick_axes[0].plot(ensure_lag, consumption_kernels[0], color="tab:red", linestyle=":", linewidth=2, label="First-period consumption component")
total_lick_axes[0].plot(ensure_lag, total_reward_lick_kernels[0], color="black", linewidth=2.5, label="Total first-period reward lick")
total_lick_axes[0].set_title("How the total reward-lick kernel is formed")
for kernel, name, color in zip(total_reward_lick_kernels, group_names, group_colors): total_lick_axes[1].plot(ensure_lag, kernel, color=color, linewidth=2, label=name)
total_lick_axes[1].set_title("Total predicted response per reward-associated lick")
for axis in total_lick_axes:
    axis.axhline(0, color="black", linewidth=0.8); axis.set_xlabel("Lag from lick (s)"); axis.legend()
total_lick_axes[0].set_ylabel("Delta tau contribution per lick (ns)")
total_lick_figure.suptitle("Ordinary lick + consumption-specific component (first Ensure TTL term excluded)")
plt.show()
first_ablation_figure, first_ablation_axis = plt.subplots(figsize=(7, 5), constrained_layout=True)
first_ablation_values = np.asarray([row["cross_validated_r2"] for row in first_transition_ablation_results])
first_ablation_bars = first_ablation_axis.bar(np.arange(2), first_ablation_values, color=["0.6", "tab:red"])
first_ablation_axis.axhline(0, color="black", linewidth=0.8)
first_ablation_axis.set_xticks(np.arange(2), ["Without first transition", "With first transition"])
first_ablation_axis.set_ylabel(f"Held-{cv_description}-out R2")
first_ablation_axis.set_title(f"Unique first-transition value: delta R2 = {first_transition_delta_r2:+.4f}")
for bar, value in zip(first_ablation_bars, first_ablation_values): first_ablation_axis.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom" if value >= 0 else "top")
plt.show()
predicted_recording_traces = {name: [] for name in group_names}; predicted_trace_mice = {name: [] for name in group_names}; row_start = 0
for recording_index, outcome in enumerate(outcomes):
    row_end = row_start + outcome.size; prediction = constrained_prediction[row_start:row_end]
    event_indices = np.flatnonzero(np.nan_to_num(ensure_matrix[recording_index, :outcome.size]) > 0); event_groups = delivery_group_indices(event_indices.size); per_group = {name: [] for name in group_names}
    for event_index, group_index in zip(event_indices, event_groups):
        if event_index - baseline_start < 0 or event_index + post_samples > prediction.size: continue
        local_baseline = prediction[event_index - baseline_start:event_index - baseline_end]; per_group[group_names[group_index]].append(prediction[event_index:event_index + post_samples] - np.nanmean(local_baseline))
    for name in group_names:
        if per_group[name]: predicted_recording_traces[name].append(np.nanmean(np.vstack(per_group[name]), axis=0)); predicted_trace_mice[name].append(mice[recording_index])
    row_start = row_end
comparison_figure, comparison_axis = plt.subplots(figsize=(10, 6), constrained_layout=True)
for name, color in zip(group_names, group_colors):
    if name not in empirical_summaries or not predicted_recording_traces[name]: continue
    _, predicted_mean, _ = mouse_balanced_summary(predicted_recording_traces[name], predicted_trace_mice[name]); empirical_mean = empirical_summaries[name]["mean"]
    comparison_axis.plot(empirical_time, empirical_mean, color=color, linewidth=2, label=f"{name} empirical"); comparison_axis.plot(empirical_time, predicted_mean, color=color, linestyle="--", linewidth=2, label=f"{name} constrained model")
comparison_axis.axhline(0, color="black", linewidth=0.8); comparison_axis.axvline(0, color="tab:orange", linestyle="--"); comparison_axis.set(title="Empirical response versus two-stage constrained model", xlabel="Time from Ensure delivery (s)", ylabel="Locally baseline-corrected delta tau (ns)"); comparison_axis.legend(ncol=2)
plt.show()
constrained_figure.savefig(OUTPUT_FOLDER / "two_stage_constrained_glm.png", dpi=200); comparison_figure.savefig(OUTPUT_FOLDER / "two_stage_empirical_comparison.png", dpi=200); first_ablation_figure.savefig(OUTPUT_FOLDER / "first_transition_ablation.png", dpi=200); total_lick_figure.savefig(OUTPUT_FOLDER / "total_reward_associated_lick_kernels.png", dpi=200)
with (OUTPUT_FOLDER / "two_stage_constrained_validation.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(constrained_results[0])); writer.writeheader(); writer.writerows(constrained_results)
with (OUTPUT_FOLDER / "first_transition_ablation.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(first_transition_ablation_results[0])); writer.writeheader(); writer.writerows(first_transition_ablation_results)
np.savez_compressed(OUTPUT_FOLDER / "two_stage_constrained_results.npz", lick_lag_s=lick_lag, frozen_lick_kernel_ns=frozen_lick_kernel, transition_lag_s=ensure_lag, first_transition_kernel_ns=first_transition_kernel, consumption_lag_s=ensure_lag, consumption_kernel_first_ns=consumption_kernels[0], consumption_kernel_early_subsequent_ns=consumption_kernels[1], consumption_kernel_middle_subsequent_ns=consumption_kernels[2], consumption_kernel_late_subsequent_ns=consumption_kernels[3], total_reward_lick_kernel_first_ns=total_reward_lick_kernels[0], total_reward_lick_kernel_early_subsequent_ns=total_reward_lick_kernels[1], total_reward_lick_kernel_middle_subsequent_ns=total_reward_lick_kernels[2], total_reward_lick_kernel_late_subsequent_ns=total_reward_lick_kernels[3], coefficients=constrained_coefficients)
print(f"Saved two-stage constrained analysis to {OUTPUT_FOLDER.resolve()}")

## Combined kernel summary

These panels keep the model components separate: dry/no-Ensure licking, the conditional Ensure-TTL term from the original joint GLM, the explicit first-delivery transition and reward-associated licking from the constrained model, and the directly observed Ensure-aligned response. The original TTL kernel is retained as a diagnostic benchmark: it shows what the joint model assigns to delivery timing after holding licking constant, not a preferred physiological impulse response. A shared absolute y-axis exposes relative magnitudes, although per-lick, per-TTL, and empirical total responses are not identical estimands.

In [ ]:
KERNEL_SUMMARY_SHARED_Y_AXIS = True
kernel_summary_figure, kernel_axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
kernel_axes[0, 0].plot(lick_lag, frozen_lick_kernel, color="tab:green", linewidth=2)
kernel_axes[0, 0].set(title="No-Ensure lick kernel", xlabel="Lag from lick (s)", ylabel="Delta tau contribution per lick (ns)")
for kernel, name, color in zip(ensure_kernels, group_names, group_colors):
    kernel_axes[0, 1].plot(ensure_lag, kernel, color=color, linewidth=2, label=name)
kernel_axes[0, 1].set(title="Original joint-GLM Ensure-TTL kernel", xlabel="Lag from Ensure TTL (s)", ylabel="Conditional delta tau contribution (ns)"); kernel_axes[0, 1].legend()
for kernel, name, color in zip(consumption_kernels, group_names, group_colors):
    kernel_axes[1, 0].plot(ensure_lag, kernel, color=color, linewidth=2, label=f"{name} consumption")
kernel_axes[1, 0].plot(ensure_lag, first_transition_kernel, color="tab:red", linestyle="--", linewidth=2, label="First Ensure TTL transition")
kernel_axes[1, 0].set(title="First transition and reward-associated consumption", xlabel="Lag (s)", ylabel="Conditional delta tau contribution (ns)"); kernel_axes[1, 0].legend()
for name, color in zip(group_names, group_colors):
    if name not in empirical_summaries: continue
    summary = empirical_summaries[name]
    kernel_axes[1, 1].plot(empirical_time, summary["mean"], color=color, linewidth=2, label=name)
    kernel_axes[1, 1].fill_between(empirical_time, summary["mean"] - summary["sem"], summary["mean"] + summary["sem"], color=color, alpha=0.2)
kernel_axes[1, 1].set(title="Empirical Ensure-aligned response", xlabel="Time from Ensure delivery (s)", ylabel="Locally baseline-corrected delta tau (ns)"); kernel_axes[1, 1].legend()
for axis in kernel_axes.flat:
    axis.axhline(0, color="black", linewidth=0.8)
if KERNEL_SUMMARY_SHARED_Y_AXIS:
    empirical_bounds = [summary["mean"] - summary["sem"] for summary in empirical_summaries.values()] + [summary["mean"] + summary["sem"] for summary in empirical_summaries.values()]
    absolute_limit = 1.05 * np.nanmax(np.abs(np.concatenate([frozen_lick_kernel, first_transition_kernel, *ensure_kernels, *consumption_kernels, *empirical_bounds])))
    for axis in kernel_axes.flat: axis.set_ylim(-absolute_limit, absolute_limit)
kernel_summary_figure.suptitle("Comparison of lick, Ensure, and consumption response components")
plt.show()
kernel_summary_figure.savefig(OUTPUT_FOLDER / "kernel_component_summary.png", dpi=200)
print(OUTPUT_FOLDER / "kernel_component_summary.png")

## Fixed long-kernel accumulation test

This test asks whether a single, linearly superimposed consumption response with a longer memory can explain the apparent delivery-order adaptation. It compares a fixed 45-second kernel, a fixed 120-second kernel, and the 45-second adaptive model using identical held-mouse-out folds. All three omit the unsupported first-Ensure TTL term. If the long fixed kernel matches the adaptive model, overlapping slow responses are sufficient and there is little evidence that later licks require different kernels.

In [ ]:
LONG_CONSUMPTION_KERNEL_S = 120.0
LONG_CONSUMPTION_BASIS_COUNT = 24
long_consumption_lag, long_consumption_basis = raised_cosine_basis(LONG_CONSUMPTION_KERNEL_S, LONG_CONSUMPTION_BASIS_COUNT, bin_width_s)
long_fixed_designs = []; row_start = 0
for recording_index, outcome in enumerate(outcomes):
    row_end = row_start + outcome.size
    valid = np.isfinite(time_matrix[recording_index]) & np.isfinite(delta_tau_matrix[recording_index])
    licks = np.nan_to_num(lick_matrix[recording_index, valid]); ensures = np.nan_to_num(ensure_matrix[recording_index, valid])
    reward_available = np.cumsum(ensures > 0) > 0
    reward_licks = licks * reward_available
    long_fixed_designs.append(np.column_stack([X[row_start:row_end, :3], convolve_basis(reward_licks, long_consumption_basis)]))
    row_start = row_end
long_fixed_X = np.vstack(long_fixed_designs)
long_fixed_columns = np.arange(long_fixed_X.shape[1])
fixed_45_columns = np.concatenate([np.arange(3), np.arange(3 + ENSURE_BASIS_COUNT, 3 + 2 * ENSURE_BASIS_COUNT)])

def long_fixed_cv(alpha):
    fold_scores = []
    for held_out in unique_groups:
        train, test = group != held_out, group == held_out; dry_train = train & no_ensure_samples
        if not np.any(dry_train) or not np.any(test): continue
        fold_dry = fit_ridge_raw(X, y, dry_model_columns, best_dry_alpha, dry_train)
        fold_lick_prediction = X[:, lick_columns] @ fold_dry[3:]
        coefficients = fit_ridge_raw(long_fixed_X, y - fold_lick_prediction, long_fixed_columns, alpha, train)
        prediction = fold_lick_prediction[test] + long_fixed_X[test] @ coefficients
        denominator = np.sum((y[test] - np.mean(y[test])) ** 2)
        if denominator > 0: fold_scores.append(1 - np.sum((y[test] - prediction) ** 2) / denominator)
    return np.nan if not fold_scores else float(np.mean(fold_scores))

fixed_45_scores = np.asarray([constrained_cv(fixed_45_columns, alpha) for alpha in RIDGE_ALPHAS])
long_fixed_scores = np.asarray([long_fixed_cv(alpha) for alpha in RIDGE_ALPHAS])
fixed_45_best_index = int(np.nanargmax(fixed_45_scores)); long_fixed_best_index = int(np.nanargmax(long_fixed_scores))
accumulation_results = [
    {"model": "Fixed 45 s", "best_alpha": float(RIDGE_ALPHAS[fixed_45_best_index]), "cross_validated_r2": float(fixed_45_scores[fixed_45_best_index])},
    {"model": "Fixed 120 s", "best_alpha": float(RIDGE_ALPHAS[long_fixed_best_index]), "cross_validated_r2": float(long_fixed_scores[long_fixed_best_index])},
    {"model": "Adaptive 45 s", "best_alpha": float(RIDGE_ALPHAS[without_first_best_index]), "cross_validated_r2": float(without_first_scores[without_first_best_index])},
]
for result in accumulation_results: print(result)
long_fixed_coefficients = fit_ridge_raw(long_fixed_X, y - frozen_lick_prediction, long_fixed_columns, accumulation_results[1]["best_alpha"], np.ones(y.size, dtype=bool))
long_consumption_kernel = long_consumption_basis @ long_fixed_coefficients[3:]
ordinary_lick_on_long_lag = np.interp(long_consumption_lag, lick_lag, frozen_lick_kernel, left=0.0, right=0.0)
long_total_reward_lick_kernel = ordinary_lick_on_long_lag + long_consumption_kernel

accumulation_figure, accumulation_axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
accumulation_values = np.asarray([result["cross_validated_r2"] for result in accumulation_results])
bars = accumulation_axes[0].bar(np.arange(3), accumulation_values, color=["0.6", "tab:purple", "tab:blue"])
accumulation_axes[0].axhline(0, color="black", linewidth=0.8); accumulation_axes[0].set_xticks(np.arange(3), [result["model"] for result in accumulation_results], rotation=15, ha="right")
accumulation_axes[0].set(title="Does a longer fixed kernel replace adaptation?", ylabel=f"Held-{cv_description}-out R2")
for bar, value in zip(bars, accumulation_values): accumulation_axes[0].text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom" if value >= 0 else "top")
accumulation_axes[1].plot(long_consumption_lag, ordinary_lick_on_long_lag, color="tab:green", linestyle="--", label="Ordinary lick component")
accumulation_axes[1].plot(long_consumption_lag, long_consumption_kernel, color="tab:purple", linestyle=":", linewidth=2, label="Fixed consumption component")
accumulation_axes[1].plot(long_consumption_lag, long_total_reward_lick_kernel, color="black", linewidth=2.5, label="Total reward-associated lick")
accumulation_axes[1].axhline(0, color="black", linewidth=0.8); accumulation_axes[1].set(title="Fixed 120-second reward-lick kernel", xlabel="Lag from lick (s)", ylabel="Delta tau contribution per lick (ns)"); accumulation_axes[1].legend()
plt.show()
delta_long_vs_short = accumulation_values[1] - accumulation_values[0]; delta_adaptive_vs_long = accumulation_values[2] - accumulation_values[1]
print(f"Long fixed versus short fixed: delta R2 = {delta_long_vs_short:+.4f}")
print(f"Adaptive versus long fixed: delta R2 = {delta_adaptive_vs_long:+.4f}")
accumulation_figure.savefig(OUTPUT_FOLDER / "fixed_long_kernel_accumulation_test.png", dpi=200)
with (OUTPUT_FOLDER / "fixed_long_kernel_accumulation_test.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(accumulation_results[0])); writer.writeheader(); writer.writerows(accumulation_results)
np.savez_compressed(OUTPUT_FOLDER / "fixed_long_kernel_accumulation_test.npz", lag_s=long_consumption_lag, ordinary_lick_kernel_ns=ordinary_lick_on_long_lag, fixed_consumption_kernel_ns=long_consumption_kernel, total_reward_lick_kernel_ns=long_total_reward_lick_kernel, ridge_alphas=RIDGE_ALPHAS, fixed_45_cv_scores=fixed_45_scores, fixed_120_cv_scores=long_fixed_scores, adaptive_45_cv_scores=without_first_scores)


## Nested duration sweep

Kernel duration and ridge strength are selected using only the training mice inside each outer held-mouse-out fold. The primary fixed-consumption model uses an intercept but no linear or quadratic time terms. All durations use 24 raised-cosine basis functions, keeping parameter count constant while changing the modeled memory. Because every duration is evaluated on the same held-out mice, the final recommendation uses a paired one-standard-error rule: select the shortest duration whose mean within-mouse loss relative to the best duration is no greater than the standard error of that paired loss.

In [ ]:
DURATION_CANDIDATES_S = np.asarray([30.0, 45.0, 60.0, 90.0, 120.0, 180.0, 240.0, 300.0, 360.0, 480.0])
DURATION_SWEEP_BASIS_COUNT = 24
group_masks = {label: group == label for label in unique_groups}
group_denominators = {label: float(np.sum((y[mask] - np.mean(y[mask])) ** 2)) for label, mask in group_masks.items()}
def fit_from_sufficient_statistics(xtx, xty, sample_count, alpha):
    scale = np.sqrt(np.maximum(np.diag(xtx) / max(sample_count, 1), 0.0)); scale[scale == 0] = 1.0
    scaled_xtx = xtx / np.outer(scale, scale); scaled_xty = xty / scale
    penalty = np.eye(xtx.shape[0]); penalty[0, 0] = 0.0
    return np.linalg.solve(scaled_xtx + alpha * penalty, scaled_xty) / scale

duration_sweep_records = []
for duration_index, duration_s in enumerate(DURATION_CANDIDATES_S, start=1):
    sweep_start = perf_counter(); sweep_lag, sweep_basis = raised_cosine_basis(duration_s, DURATION_SWEEP_BASIS_COUNT, bin_width_s)
    sweep_designs = []; row_start = 0
    for recording_index, outcome in enumerate(outcomes):
        row_end = row_start + outcome.size
        valid = np.isfinite(time_matrix[recording_index]) & np.isfinite(delta_tau_matrix[recording_index])
        licks = np.nan_to_num(lick_matrix[recording_index, valid]); ensures = np.nan_to_num(ensure_matrix[recording_index, valid])
        reward_licks = licks * (np.cumsum(ensures > 0) > 0)
        sweep_designs.append(np.column_stack([X[row_start:row_end, [0]], convolve_basis(reward_licks, sweep_basis)]))
        row_start = row_end
    sweep_X = np.vstack(sweep_designs)
    xtx_by_group = {label: sweep_X[mask].T @ sweep_X[mask] for label, mask in group_masks.items()}
    sample_count_by_group = {label: int(np.sum(mask)) for label, mask in group_masks.items()}
    for outer_label in unique_groups:
        residual = y
        xty_by_group = {label: sweep_X[mask].T @ residual[mask] for label, mask in group_masks.items()}
        residual_square_by_group = {label: float(residual[mask] @ residual[mask]) for label, mask in group_masks.items()}
        training_labels = [label for label in unique_groups if label != outer_label]
        alpha_means, alpha_ses = [], []
        for alpha in RIDGE_ALPHAS:
            inner_scores = []
            for inner_label in training_labels:
                fit_labels = [label for label in training_labels if label != inner_label]
                train_xtx = np.sum(np.stack([xtx_by_group[label] for label in fit_labels]), axis=0)
                train_xty = np.sum(np.stack([xty_by_group[label] for label in fit_labels]), axis=0)
                train_count = sum(sample_count_by_group[label] for label in fit_labels)
                coefficients = fit_from_sufficient_statistics(train_xtx, train_xty, train_count, alpha)
                inner_sse = residual_square_by_group[inner_label] - 2 * coefficients @ xty_by_group[inner_label] + coefficients @ xtx_by_group[inner_label] @ coefficients
                if group_denominators[inner_label] > 0: inner_scores.append(1 - inner_sse / group_denominators[inner_label])
            alpha_means.append(np.nanmean(inner_scores)); alpha_ses.append(np.nanstd(inner_scores, ddof=1) / np.sqrt(len(inner_scores)) if len(inner_scores) > 1 else np.nan)
        best_alpha_index = int(np.nanargmax(alpha_means)); selected_alpha = float(RIDGE_ALPHAS[best_alpha_index])
        outer_train_xtx = np.sum(np.stack([xtx_by_group[label] for label in training_labels]), axis=0)
        outer_train_xty = np.sum(np.stack([xty_by_group[label] for label in training_labels]), axis=0)
        outer_train_count = sum(sample_count_by_group[label] for label in training_labels)
        outer_coefficients = fit_from_sufficient_statistics(outer_train_xtx, outer_train_xty, outer_train_count, selected_alpha)
        outer_sse = residual_square_by_group[outer_label] - 2 * outer_coefficients @ xty_by_group[outer_label] + outer_coefficients @ xtx_by_group[outer_label] @ outer_coefficients
        outer_r2 = np.nan if group_denominators[outer_label] <= 0 else 1 - outer_sse / group_denominators[outer_label]
        duration_sweep_records.append({"held_out_mouse": str(outer_label), "duration_s": float(duration_s), "basis_count": DURATION_SWEEP_BASIS_COUNT, "selected_alpha": selected_alpha, "inner_mean_r2": float(alpha_means[best_alpha_index]), "inner_se_r2": float(alpha_ses[best_alpha_index]), "outer_r2": float(outer_r2)})
    print(f"[{duration_index}/{DURATION_CANDIDATES_S.size}] {duration_s:.0f} s complete in {perf_counter() - sweep_start:.1f} s")

outer_score_matrix = np.asarray([[next(row["outer_r2"] for row in duration_sweep_records if row["held_out_mouse"] == str(label) and row["duration_s"] == duration) for duration in DURATION_CANDIDATES_S] for label in unique_groups])
duration_mean_r2 = np.mean(outer_score_matrix, axis=0); duration_se_r2 = np.std(outer_score_matrix, axis=0, ddof=1) / np.sqrt(unique_groups.size)
best_duration_index = int(np.nanargmax(duration_mean_r2)); paired_loss = outer_score_matrix[:, [best_duration_index]] - outer_score_matrix
paired_loss_mean = np.mean(paired_loss, axis=0); paired_loss_se = np.std(paired_loss, axis=0, ddof=1) / np.sqrt(unique_groups.size)
paired_one_se_eligible = paired_loss_mean <= paired_loss_se
recommended_duration = float(DURATION_CANDIDATES_S[np.flatnonzero(paired_one_se_eligible)[0]])
nested_selected_records = []
for outer_label in unique_groups:
    candidates = [row for row in duration_sweep_records if row["held_out_mouse"] == str(outer_label)]
    best_candidate = max(candidates, key=lambda row: row["inner_mean_r2"]); threshold = best_candidate["inner_mean_r2"] - best_candidate["inner_se_r2"]
    nested_selected_records.append(min((row for row in candidates if row["inner_mean_r2"] >= threshold), key=lambda row: row["duration_s"]))
nested_selector_r2 = np.asarray([row["outer_r2"] for row in nested_selected_records])
print(f"Recommended duration by paired one-SE rule: {recommended_duration:.0f} s")
print(f"Fully nested duration-selector R2: {np.mean(nested_selector_r2):.3f} +/- {np.std(nested_selector_r2, ddof=1) / np.sqrt(nested_selector_r2.size):.3f} SEM")

duration_figure, duration_axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
duration_axes[0].errorbar(DURATION_CANDIDATES_S, duration_mean_r2, yerr=duration_se_r2, marker="o", capsize=4, color="tab:blue")
duration_axes[0].axvline(recommended_duration, color="tab:orange", linestyle="--", label=f"Paired one-SE recommendation: {recommended_duration:.0f} s")
duration_axes[0].set(title="Fixed-kernel duration sweep", xlabel="Kernel duration (s)", ylabel=f"Held-{cv_description}-out R2"); duration_axes[0].legend()
for outer_label in unique_groups:
    mouse_scores = [row["outer_r2"] for row in duration_sweep_records if row["held_out_mouse"] == str(outer_label)]
    duration_axes[1].plot(DURATION_CANDIDATES_S, mouse_scores, marker="o", alpha=0.75, label=str(outer_label))
duration_axes[1].axvline(recommended_duration, color="tab:orange", linestyle="--"); duration_axes[1].set(title="Performance for each held-out mouse", xlabel="Kernel duration (s)", ylabel="Outer-fold R2"); duration_axes[1].legend(ncol=2, fontsize=8)
plt.show()
duration_figure.savefig(OUTPUT_FOLDER / "nested_fixed_kernel_duration_sweep.png", dpi=200)
with (OUTPUT_FOLDER / "nested_fixed_kernel_duration_sweep.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(duration_sweep_records[0])); writer.writeheader(); writer.writerows(duration_sweep_records)
with (OUTPUT_FOLDER / "nested_duration_selected_by_mouse.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(nested_selected_records[0])); writer.writeheader(); writer.writerows(nested_selected_records)


## Final model at the selected duration

This section refits the simple intercept-only, consumption-only model at the duration recommended above. It excludes the ordinary 10-second lick kernel as well as linear and quadratic time terms; the ablations below retain the evidence for those decisions. The displayed full-data kernel is descriptive; the example prediction is genuinely held-mouse-out, with ridge strength selected inside each training fold. Because the duration was chosen using this cohort, confirmation in additional mice remains important.

In [ ]:
SELECTED_CONSUMPTION_KERNEL_S = recommended_duration
selected_lag, selected_basis = raised_cosine_basis(SELECTED_CONSUMPTION_KERNEL_S, DURATION_SWEEP_BASIS_COUNT, bin_width_s)
selected_designs = []
row_start = 0
for recording_index, outcome in enumerate(outcomes):
    row_end = row_start + outcome.size
    valid = np.isfinite(time_matrix[recording_index]) & np.isfinite(delta_tau_matrix[recording_index])
    licks = np.nan_to_num(lick_matrix[recording_index, valid])
    ensures = np.nan_to_num(ensure_matrix[recording_index, valid])
    reward_licks = licks * (np.cumsum(ensures > 0) > 0)
    selected_designs.append(np.column_stack([X[row_start:row_end, [0]], convolve_basis(reward_licks, selected_basis)]))
    row_start = row_end
selected_X = np.vstack(selected_designs)
selected_columns = np.arange(selected_X.shape[1])
selected_duration_rows = [row for row in duration_sweep_records if row["duration_s"] == SELECTED_CONSUMPTION_KERNEL_S]
selected_final_alpha = float(np.median([row["selected_alpha"] for row in selected_duration_rows]))
selected_duration_index = int(np.flatnonzero(DURATION_CANDIDATES_S == SELECTED_CONSUMPTION_KERNEL_S)[0])
selected_cv_r2 = float(duration_mean_r2[selected_duration_index])
selected_coefficients = fit_ridge_raw(selected_X, y, selected_columns, selected_final_alpha, np.ones(y.size, dtype=bool))
selected_consumption_kernel = selected_basis @ selected_coefficients[1:]
selected_total_reward_lick_kernel = selected_consumption_kernel

# Build a genuinely held-mouse-out prediction at the selected duration.
selected_oof_prediction = np.full(y.size, np.nan)
for outer_label in unique_groups:
    train, test = group != outer_label, group == outer_label
    fold_alpha = next(row["selected_alpha"] for row in selected_duration_rows if row["held_out_mouse"] == str(outer_label))
    fold_coefficients = fit_ridge_raw(selected_X, y, selected_columns, fold_alpha, train)
    selected_oof_prediction[test] = selected_X[test] @ fold_coefficients

recording_offsets = np.concatenate([[0], np.cumsum([outcome.size for outcome in outcomes])])
example_start, example_end = recording_offsets[example_index], recording_offsets[example_index + 1]
selected_example_prediction = selected_oof_prediction[example_start:example_end]

selected_figure, selected_axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
selected_axes[0, 0].plot(selected_lag, selected_consumption_kernel, color="tab:purple", linewidth=2)
selected_axes[0, 0].axhline(0, color="black", linewidth=0.8)
selected_axes[0, 0].set(title=f"Reward-associated consumption kernel ({SELECTED_CONSUMPTION_KERNEL_S:.0f} s)", xlabel="Lag from reward-associated lick (s)", ylabel="Delta tau contribution per lick (ns)")
early_kernel_mask = selected_lag <= 60.0
selected_axes[0, 1].plot(selected_lag[early_kernel_mask], selected_consumption_kernel[early_kernel_mask], color="tab:purple", linewidth=2)
selected_axes[0, 1].axhline(0, color="black", linewidth=0.8)
selected_axes[0, 1].set(title="Early kernel detail", xlabel="Lag from reward-associated lick (s)", ylabel="Delta tau contribution per lick (ns)")
selected_axes[1, 0].errorbar(DURATION_CANDIDATES_S, duration_mean_r2, yerr=duration_se_r2, marker="o", capsize=4, color="tab:blue")
selected_axes[1, 0].axvline(SELECTED_CONSUMPTION_KERNEL_S, color="tab:orange", linestyle="--", label=f"Selected: {SELECTED_CONSUMPTION_KERNEL_S:.0f} s")
selected_axes[1, 0].set(title="Why this duration was selected", xlabel="Kernel duration (s)", ylabel=f"Held-{cv_description}-out R2"); selected_axes[1, 0].legend()
selected_axes[1, 1].plot(example_time, example_outcome, color="0.75", linewidth=0.8, label="Observed")
selected_axes[1, 1].plot(example_time, selected_example_prediction, color="tab:blue", linewidth=1.2, label="Held-mouse-out prediction")
selected_axes[1, 1].set(title=f"Example: {mice[example_index]} {recording_names[example_index]}", xlabel="Recording time (s)", ylabel="Delta tau (ns)"); selected_axes[1, 1].legend()
selected_figure.suptitle(f"Primary consumption-only model: {SELECTED_CONSUMPTION_KERNEL_S:.0f}-second memory")
plt.show()

selected_figure.savefig(OUTPUT_FOLDER / "selected_fixed_consumption_model.png", dpi=200)
np.savez_compressed(OUTPUT_FOLDER / "selected_fixed_consumption_model.npz", duration_s=np.asarray(SELECTED_CONSUMPTION_KERNEL_S), lag_s=selected_lag, consumption_kernel_ns=selected_consumption_kernel, reward_associated_kernel_ns=selected_total_reward_lick_kernel, best_alpha=np.asarray(selected_final_alpha), cross_validated_r2=np.asarray(selected_cv_r2), coefficients=selected_coefficients, held_out_prediction_ns=selected_oof_prediction)
with (OUTPUT_FOLDER / "selected_fixed_consumption_model_summary.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["duration_s", "basis_count", "drift_terms", "ordinary_lick_kernel", "final_alpha", "cross_validated_r2"]); writer.writeheader(); writer.writerow({"duration_s": SELECTED_CONSUMPTION_KERNEL_S, "basis_count": DURATION_SWEEP_BASIS_COUNT, "drift_terms": "intercept only", "ordinary_lick_kernel": "excluded", "final_alpha": selected_final_alpha, "cross_validated_r2": selected_cv_r2})
print(f"Final selected model: {SELECTED_CONSUMPTION_KERNEL_S:.0f} s, alpha={selected_final_alpha:g}, held-{cv_description}-out R2={selected_cv_r2:.3f}")
print(OUTPUT_FOLDER / "selected_fixed_consumption_model.png")


## Cohort mean: empirical signal versus selected model

Ensure recordings are aligned to the first delivery, baseline-corrected to the 60-to-10-second pre-delivery interval, averaged within each mouse, and then averaged equally across mice. The modeled trace uses held-mouse-out predictions, so the comparison does not use each mouse to fit its own prediction.

In [ ]:
COHORT_PRE_S = 60.0
COHORT_BASELINE_S = (-60.0, -10.0)
COHORT_MAX_POST_S = 600.0
ensure_recording_indices = [index for index in range(mice.size) if np.any(np.nan_to_num(ensure_matrix[index]) > 0)]
first_event_indices = {index: int(np.flatnonzero(np.nan_to_num(ensure_matrix[index]) > 0)[0]) for index in ensure_recording_indices}
pre_samples = int(round(COHORT_PRE_S / bin_width_s))
available_post_samples = [outcomes[index].size - first_event_indices[index] for index in ensure_recording_indices if first_event_indices[index] >= pre_samples]
cohort_post_samples = min(int(round(COHORT_MAX_POST_S / bin_width_s)), min(available_post_samples))
cohort_time = np.arange(-pre_samples, cohort_post_samples) * bin_width_s
baseline_mask = (cohort_time >= COHORT_BASELINE_S[0]) & (cohort_time < COHORT_BASELINE_S[1])
cohort_empirical_recordings, cohort_model_recordings, cohort_recording_mice = [], [], []
for recording_index in ensure_recording_indices:
    event_index = first_event_indices[recording_index]
    if event_index < pre_samples or event_index + cohort_post_samples > outcomes[recording_index].size:
        continue
    local_slice = slice(event_index - pre_samples, event_index + cohort_post_samples)
    empirical_trace = outcomes[recording_index][local_slice].copy()
    flat_slice = slice(recording_offsets[recording_index] + event_index - pre_samples, recording_offsets[recording_index] + event_index + cohort_post_samples)
    model_trace = selected_oof_prediction[flat_slice].copy()
    empirical_trace -= np.nanmean(empirical_trace[baseline_mask])
    model_trace -= np.nanmean(model_trace[baseline_mask])
    cohort_empirical_recordings.append(empirical_trace); cohort_model_recordings.append(model_trace); cohort_recording_mice.append(mice[recording_index])
cohort_empirical_recordings = np.vstack(cohort_empirical_recordings); cohort_model_recordings = np.vstack(cohort_model_recordings); cohort_recording_mice = np.asarray(cohort_recording_mice)
cohort_mouse_labels = np.unique(cohort_recording_mice)
cohort_empirical_mouse_means = np.vstack([np.nanmean(cohort_empirical_recordings[cohort_recording_mice == mouse], axis=0) for mouse in cohort_mouse_labels])
cohort_model_mouse_means = np.vstack([np.nanmean(cohort_model_recordings[cohort_recording_mice == mouse], axis=0) for mouse in cohort_mouse_labels])
cohort_empirical_mean = np.nanmean(cohort_empirical_mouse_means, axis=0); cohort_model_mean = np.nanmean(cohort_model_mouse_means, axis=0)
cohort_empirical_sem = np.nanstd(cohort_empirical_mouse_means, axis=0, ddof=1) / np.sqrt(cohort_mouse_labels.size)
cohort_model_sem = np.nanstd(cohort_model_mouse_means, axis=0, ddof=1) / np.sqrt(cohort_mouse_labels.size)

cohort_figure, cohort_axis = plt.subplots(figsize=(13, 6), constrained_layout=True)
for mouse_trace in cohort_empirical_mouse_means: cohort_axis.plot(cohort_time, mouse_trace, color="0.75", linewidth=0.8, alpha=0.65)
for mouse_trace in cohort_model_mouse_means: cohort_axis.plot(cohort_time, mouse_trace, color="tab:blue", linewidth=0.7, alpha=0.18)
cohort_axis.fill_between(cohort_time, cohort_empirical_mean - cohort_empirical_sem, cohort_empirical_mean + cohort_empirical_sem, color="0.35", alpha=0.18)
cohort_axis.fill_between(cohort_time, cohort_model_mean - cohort_model_sem, cohort_model_mean + cohort_model_sem, color="tab:blue", alpha=0.18)
cohort_axis.plot(cohort_time, cohort_empirical_mean, color="black", linewidth=2.5, label=f"Empirical mouse mean (n={cohort_mouse_labels.size})")
cohort_axis.plot(cohort_time, cohort_model_mean, color="tab:blue", linewidth=2.5, label=f"{SELECTED_CONSUMPTION_KERNEL_S:.0f}-s model, held-mouse-out")
cohort_axis.axhline(0, color="black", linewidth=0.8); cohort_axis.axvline(0, color="tab:orange", linestyle="--", label="First Ensure")
cohort_axis.set(title="Mouse-balanced empirical and modeled signals after first Ensure", xlabel="Time from first Ensure delivery (s)", ylabel="Baseline-corrected delta tau (ns)"); cohort_axis.legend()
plt.show()
cohort_figure.savefig(OUTPUT_FOLDER / "selected_model_cohort_mean.png", dpi=200)
np.savez_compressed(OUTPUT_FOLDER / "selected_model_cohort_mean.npz", time_s=cohort_time, mouse=cohort_mouse_labels, empirical_mouse_mean_ns=cohort_empirical_mouse_means, model_mouse_mean_ns=cohort_model_mouse_means, empirical_mean_ns=cohort_empirical_mean, empirical_sem_ns=cohort_empirical_sem, model_mean_ns=cohort_model_mean, model_sem_ns=cohort_model_sem)
print(f"Cohort comparison includes {cohort_mouse_labels.size} mice and {cohort_empirical_recordings.shape[0]} Ensure recordings")
print(OUTPUT_FOLDER / "selected_model_cohort_mean.png")


## Are the linear and quadratic time terms needed?

This nested held-mouse-out ablation keeps the selected consumption duration fixed and compares three nuisance-drift specifications: intercept only, intercept plus linear time, and intercept plus linear and quadratic time. Ridge strength is selected using only the training mice within each outer fold. Predictive performance and kernel stability are evaluated together.

In [ ]:
drift_ablation_X = np.column_stack([X[:, :3], selected_X[:, 1:]])
drift_variant_columns = {
    "Intercept only": np.concatenate([[0], np.arange(3, drift_ablation_X.shape[1])]),
    "+ linear time": np.concatenate([[0, 1], np.arange(3, drift_ablation_X.shape[1])]),
    "+ linear + quadratic time": np.arange(drift_ablation_X.shape[1]),
}
drift_variant_results, drift_variant_kernels, drift_variant_total_kernels = [], {}, {}
drift_variant_outer_scores = {}
for variant_name, source_columns in drift_variant_columns.items():
    variant_X = drift_ablation_X[:, source_columns]
    variant_group_xtx = {label: variant_X[mask].T @ variant_X[mask] for label, mask in group_masks.items()}
    variant_outer_records = []
    for outer_label in unique_groups:
        residual = y
        variant_group_xty = {label: variant_X[mask].T @ residual[mask] for label, mask in group_masks.items()}
        residual_square_by_group = {label: float(residual[mask] @ residual[mask]) for label, mask in group_masks.items()}
        training_labels = [label for label in unique_groups if label != outer_label]
        alpha_means = []
        for alpha in RIDGE_ALPHAS:
            inner_scores = []
            for inner_label in training_labels:
                fit_labels = [label for label in training_labels if label != inner_label]
                train_xtx = np.sum(np.stack([variant_group_xtx[label] for label in fit_labels]), axis=0)
                train_xty = np.sum(np.stack([variant_group_xty[label] for label in fit_labels]), axis=0)
                train_count = sum(sample_count_by_group[label] for label in fit_labels)
                coefficients = fit_from_sufficient_statistics(train_xtx, train_xty, train_count, alpha)
                inner_sse = residual_square_by_group[inner_label] - 2 * coefficients @ variant_group_xty[inner_label] + coefficients @ variant_group_xtx[inner_label] @ coefficients
                if group_denominators[inner_label] > 0: inner_scores.append(1 - inner_sse / group_denominators[inner_label])
            alpha_means.append(np.nanmean(inner_scores))
        selected_alpha = float(RIDGE_ALPHAS[int(np.nanargmax(alpha_means))])
        train_xtx = np.sum(np.stack([variant_group_xtx[label] for label in training_labels]), axis=0)
        train_xty = np.sum(np.stack([variant_group_xty[label] for label in training_labels]), axis=0)
        train_count = sum(sample_count_by_group[label] for label in training_labels)
        coefficients = fit_from_sufficient_statistics(train_xtx, train_xty, train_count, selected_alpha)
        outer_sse = residual_square_by_group[outer_label] - 2 * coefficients @ variant_group_xty[outer_label] + coefficients @ variant_group_xtx[outer_label] @ coefficients
        outer_r2 = 1 - outer_sse / group_denominators[outer_label]
        variant_outer_records.append({"model": variant_name, "held_out_mouse": str(outer_label), "selected_alpha": selected_alpha, "outer_r2": float(outer_r2)})
    outer_scores = np.asarray([row["outer_r2"] for row in variant_outer_records])
    drift_variant_outer_scores[variant_name] = outer_scores
    final_alpha = float(np.median([row["selected_alpha"] for row in variant_outer_records]))
    full_coefficients = fit_ridge_raw(variant_X, y, np.arange(variant_X.shape[1]), final_alpha, np.ones(y.size, dtype=bool))
    drift_count = int(np.sum(source_columns < 3))
    consumption_kernel = selected_basis @ full_coefficients[drift_count:]
    drift_variant_kernels[variant_name] = consumption_kernel
    drift_variant_total_kernels[variant_name] = consumption_kernel
    result = {"model": variant_name, "mean_outer_r2": float(np.mean(outer_scores)), "sem_outer_r2": float(np.std(outer_scores, ddof=1) / np.sqrt(outer_scores.size)), "final_alpha": final_alpha}
    drift_variant_results.append(result); print(result)

drift_names = list(drift_variant_columns)
drift_colors = ["0.45", "tab:orange", "tab:blue"]
drift_means = np.asarray([next(row["mean_outer_r2"] for row in drift_variant_results if row["model"] == name) for name in drift_names])
drift_sems = np.asarray([next(row["sem_outer_r2"] for row in drift_variant_results if row["model"] == name) for name in drift_names])
drift_figure, drift_axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
drift_axes[0].bar(np.arange(len(drift_names)), drift_means, yerr=drift_sems, capsize=4, color=drift_colors, alpha=0.85)
for index, name in enumerate(drift_names): drift_axes[0].scatter(np.full(unique_groups.size, index), drift_variant_outer_scores[name], color="black", s=20, alpha=0.65, zorder=3)
drift_axes[0].axhline(0, color="black", linewidth=0.8); drift_axes[0].set_xticks(np.arange(len(drift_names)), drift_names, rotation=15, ha="right")
drift_axes[0].set(title="Held-mouse-out predictive value", ylabel="Outer-fold R2")
for name, color in zip(drift_names, drift_colors):
    drift_axes[1].plot(selected_lag, drift_variant_kernels[name], color=color, linewidth=2, label=name)
    drift_axes[2].plot(selected_lag, drift_variant_total_kernels[name], color=color, linewidth=2, label=name)
drift_axes[1].set(title="Consumption-specific kernel stability", xlabel="Lag from reward-associated lick (s)", ylabel="Delta tau per lick (ns)")
drift_axes[2].set(title="Total reward-associated lick kernel", xlabel="Lag from lick (s)", ylabel="Delta tau per lick (ns)")
for axis in drift_axes[1:]: axis.axhline(0, color="black", linewidth=0.8); axis.legend()
drift_figure.suptitle(f"Drift-term ablation with fixed {SELECTED_CONSUMPTION_KERNEL_S:.0f}-second consumption memory")
plt.show()
drift_figure.savefig(OUTPUT_FOLDER / "selected_model_drift_ablation.png", dpi=200)
with (OUTPUT_FOLDER / "selected_model_drift_ablation.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(drift_variant_results[0])); writer.writeheader(); writer.writerows(drift_variant_results)
np.savez_compressed(OUTPUT_FOLDER / "selected_model_drift_ablation.npz", lag_s=selected_lag, model=np.asarray(drift_names), outer_r2=np.vstack([drift_variant_outer_scores[name] for name in drift_names]), consumption_kernel_ns=np.vstack([drift_variant_kernels[name] for name in drift_names]), total_reward_lick_kernel_ns=np.vstack([drift_variant_total_kernels[name] for name in drift_names]))
print(OUTPUT_FOLDER / "selected_model_drift_ablation.png")


## Is the ordinary 10-second lick kernel needed?

The primary model excludes the ordinary-lick component. This retained ablation adds the frozen 10-second kernel back and refits the selected long-duration consumption component. Ridge strength is selected within the training mice of every outer fold. The comparison documents whether the extra component improves held-mouse-out prediction or materially changes the total reward-associated response.

In [ ]:
no_ordinary_rows = [row for row in duration_sweep_records if row["duration_s"] == SELECTED_CONSUMPTION_KERNEL_S]
no_ordinary_outer_r2 = np.asarray([next(row["outer_r2"] for row in no_ordinary_rows if row["held_out_mouse"] == str(label)) for label in unique_groups])
with_ordinary_group_xtx = {label: selected_X[mask].T @ selected_X[mask] for label, mask in group_masks.items()}
with_ordinary_records = []
for outer_label in unique_groups:
    outer_train = group != outer_label; outer_dry_train = outer_train & no_ensure_samples
    outer_dry_coefficients = fit_ridge_raw(X, y, dry_model_columns, best_dry_alpha, outer_dry_train)
    outer_lick_prediction = X[:, lick_columns] @ outer_dry_coefficients[3:]
    residual = y - outer_lick_prediction
    with_ordinary_group_xty = {label: selected_X[mask].T @ residual[mask] for label, mask in group_masks.items()}
    residual_square_by_group = {label: float(residual[mask] @ residual[mask]) for label, mask in group_masks.items()}
    training_labels = [label for label in unique_groups if label != outer_label]
    alpha_means = []
    for alpha in RIDGE_ALPHAS:
        inner_scores = []
        for inner_label in training_labels:
            fit_labels = [label for label in training_labels if label != inner_label]
            train_xtx = np.sum(np.stack([with_ordinary_group_xtx[label] for label in fit_labels]), axis=0)
            train_xty = np.sum(np.stack([with_ordinary_group_xty[label] for label in fit_labels]), axis=0)
            train_count = sum(sample_count_by_group[label] for label in fit_labels)
            coefficients = fit_from_sufficient_statistics(train_xtx, train_xty, train_count, alpha)
            inner_sse = residual_square_by_group[inner_label] - 2 * coefficients @ with_ordinary_group_xty[inner_label] + coefficients @ with_ordinary_group_xtx[inner_label] @ coefficients
            if group_denominators[inner_label] > 0: inner_scores.append(1 - inner_sse / group_denominators[inner_label])
        alpha_means.append(np.nanmean(inner_scores))
    selected_alpha = float(RIDGE_ALPHAS[int(np.nanargmax(alpha_means))])
    train_xtx = np.sum(np.stack([with_ordinary_group_xtx[label] for label in training_labels]), axis=0)
    train_xty = np.sum(np.stack([with_ordinary_group_xty[label] for label in training_labels]), axis=0)
    train_count = sum(sample_count_by_group[label] for label in training_labels)
    coefficients = fit_from_sufficient_statistics(train_xtx, train_xty, train_count, selected_alpha)
    outer_sse = residual_square_by_group[outer_label] - 2 * coefficients @ with_ordinary_group_xty[outer_label] + coefficients @ with_ordinary_group_xtx[outer_label] @ coefficients
    outer_r2 = 1 - outer_sse / group_denominators[outer_label]
    with_ordinary_records.append({"held_out_mouse": str(outer_label), "selected_alpha": selected_alpha, "outer_r2": float(outer_r2)})
with_ordinary_outer_r2 = np.asarray([row["outer_r2"] for row in with_ordinary_records])
with_ordinary_final_alpha = float(np.median([row["selected_alpha"] for row in with_ordinary_records]))
with_ordinary_coefficients = fit_ridge_raw(selected_X, y - frozen_lick_prediction, selected_columns, with_ordinary_final_alpha, np.ones(y.size, dtype=bool))
with_ordinary_consumption_kernel = selected_basis @ with_ordinary_coefficients[1:]
ordinary_lick_on_selected_lag = np.interp(selected_lag, lick_lag, frozen_lick_kernel, left=0.0, right=0.0)
with_ordinary_total_kernel = ordinary_lick_on_selected_lag + with_ordinary_consumption_kernel
ordinary_ablation_names = [f"Primary: {SELECTED_CONSUMPTION_KERNEL_S:.0f}-s consumption only", "Add 10-s ordinary lick"]
ordinary_ablation_scores = [no_ordinary_outer_r2, with_ordinary_outer_r2]
ordinary_ablation_means = np.asarray([np.mean(scores) for scores in ordinary_ablation_scores])
ordinary_ablation_sems = np.asarray([np.std(scores, ddof=1) / np.sqrt(scores.size) for scores in ordinary_ablation_scores])
ordinary_ablation_delta = with_ordinary_outer_r2 - no_ordinary_outer_r2
print(f"Adding ordinary kernel: mean paired delta R2 = {np.mean(ordinary_ablation_delta):+.4f} +/- {np.std(ordinary_ablation_delta, ddof=1) / np.sqrt(ordinary_ablation_delta.size):.4f} SEM")

ordinary_figure, ordinary_axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
ordinary_axes[0].bar(np.arange(2), ordinary_ablation_means, yerr=ordinary_ablation_sems, capsize=4, color=["tab:blue", "tab:purple"], alpha=0.85)
for mouse_index in range(unique_groups.size): ordinary_axes[0].plot(np.arange(2), [scores[mouse_index] for scores in ordinary_ablation_scores], color="0.45", marker="o", linewidth=1, alpha=0.7)
ordinary_axes[0].axhline(0, color="black", linewidth=0.8); ordinary_axes[0].set_xticks(np.arange(2), ordinary_ablation_names, rotation=12, ha="right")
ordinary_axes[0].set(title=f"Predictive value of ordinary kernel\npaired delta R2 = {np.mean(ordinary_ablation_delta):+.4f}", ylabel="Held-mouse-out R2")
ordinary_axes[1].plot(selected_lag, selected_consumption_kernel, color="tab:purple", linewidth=2, label="Primary consumption-only kernel")
ordinary_axes[1].plot(selected_lag, with_ordinary_consumption_kernel, color="tab:blue", linewidth=2, label="Consumption component after ordinary lick")
ordinary_axes[1].set(title=f"Estimated {SELECTED_CONSUMPTION_KERNEL_S:.0f}-second component", xlabel="Lag from reward-associated lick (s)", ylabel="Delta tau per lick (ns)")
ordinary_axes[2].plot(selected_lag, selected_total_reward_lick_kernel, color="tab:purple", linewidth=2, label="Primary consumption-only kernel")
ordinary_axes[2].plot(selected_lag, with_ordinary_total_kernel, color="tab:blue", linewidth=2, label="Total with ordinary lick added")
ordinary_axes[2].set(title="Total predicted reward-associated response", xlabel="Lag from lick (s)", ylabel="Delta tau per lick (ns)")
for axis in ordinary_axes[1:]: axis.axhline(0, color="black", linewidth=0.8); axis.legend()
ordinary_figure.suptitle("Ablation of the frozen 10-second ordinary-lick component")
plt.show()
ordinary_figure.savefig(OUTPUT_FOLDER / "selected_model_ordinary_lick_ablation.png", dpi=200)
ordinary_ablation_summary = [
    {"model": ordinary_ablation_names[0], "mean_outer_r2": float(ordinary_ablation_means[0]), "sem_outer_r2": float(ordinary_ablation_sems[0])},
    {"model": ordinary_ablation_names[1], "mean_outer_r2": float(ordinary_ablation_means[1]), "sem_outer_r2": float(ordinary_ablation_sems[1])},
]
with (OUTPUT_FOLDER / "selected_model_ordinary_lick_ablation.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(ordinary_ablation_summary[0])); writer.writeheader(); writer.writerows(ordinary_ablation_summary)
np.savez_compressed(OUTPUT_FOLDER / "selected_model_ordinary_lick_ablation.npz", mouse=unique_groups.astype(str), no_ordinary_outer_r2=no_ordinary_outer_r2, with_ordinary_outer_r2=with_ordinary_outer_r2, lag_s=selected_lag, primary_consumption_kernel_ns=selected_consumption_kernel, ordinary_lick_kernel_ns=ordinary_lick_on_selected_lag, with_ordinary_consumption_kernel_ns=with_ordinary_consumption_kernel, with_ordinary_total_kernel_ns=with_ordinary_total_kernel)
print(OUTPUT_FOLDER / "selected_model_ordinary_lick_ablation.png")
